## General Guidelines:
> 1 GB: Stick with pandas.

> 1-10 GB: Evaluate your system's RAM and the complexity of the operations. You can still use pandas if your system has enough memory and the operations are relatively simple, but consider PySpark for future scalability.

> 10 GB: PySpark becomes necessary as pandas will struggle with memory constraints and performance. At this scale, distributed computing brings significant performance improvements.

In [1]:
from pyspark.sql import SparkSession

In [2]:
spark = SparkSession.builder \
    .appName("SparkByExamples") \
    .config("spark.some.config.option", "config-value") \
    .getOrCreate()

In [3]:
print(f"Spark Version: {spark.version}")

Spark Version: 3.5.7


In [4]:
myRange = spark.range(1000)
# myRange = spark.range(1000).toDF("number") # toDF is just for renaming
type(myRange)


,id
0,0
1,1
2,2
3,3
4,4
...,...
995,995
996,996
997,997
998,998


**Spark.DataFrame**

Spark possède plusieurs core API: Datasets, DataFrames, Tables SQL et Resilient Distributed Datasets (RDDs). Ces différents objets représentent toutes des collections de données distribuées, capable de stocker les données partitionnées sur différentes machines d’un cluster, permettant le calcul en parallèle. Les DataFrames, disponibles dans 3 langages sur 4, sont les plus simples et les plus populaires. 

Le concept de DataFrame n'est pas propre à Spark. R et Python ont tous deux des concepts similaires. Cependant, les DataFrames Python-pandas / R existent sur une seule machine plutôt que sur plusieurs machines (Dask, polar, plus léger mais moins mature).

Comme Spark possède des interfaces de langage pour Python et R, il est assez facile de convertir les DataFrames Pandas (Python) en DataFrames Spark, et les DataFrames R en DataFrames Spark. Souvent dans le sens : SparkDF => manipulation et réduction => pd dataframe. It is easy to get started with Dask DataFrame, but using it well does require some experience.

La moralité d’histoire est que : si vos données tiennent dans la RAM d’une seule machine, pandas DF est plus facile et surtout souvent plus rapide à utiliser que Spark DataFrame. [là je vais sortir la phrase fondamentale de ce module] Si les outils "Big Data" peuvent être passionnants, ils sont presque systématiquement moins bons que les outils de données normaux (ie les outils classiques de machine individuelle), tant que ces outils restent appropriés vis-à-vis au volume de données. En termes de fonctionnalités, les solutions de bigdata sont plus limitées également. (For data that fits into RAM, pandas can often be faster and easier to use than Dask DataFrame. While “Big Data” tools can be exciting, they are almost always worse than normal data tools while those remain appropriate.)

In [ ]:
myRange # no output
myRange.show(10)
myRange.toPandas()



# no need for argument, no autocompletion

**Partitions** (à partir de maintenant on illustre avec DF)
Une df spark va être divisé et chaque morceau est stocké sur une machine physique (individuelle) du cluster.
Une partition est une collection de lignes.  
Avec les DataFrames, pour la plupart du temps, vous ne manipulez pas les partitions individuellement. Vous spécifiez simplement les transformations du haut niveau, comme si vous manipuliez les données en entier.
C'est Spark qui détermine comment ce travail sera exécuté sur le cluster.



In [ ]:
myRange.rdd.getNumPartitions() # get the number of partitions. Par défaut, lorsque vous créez un DataFrame avec range, il a huit partitions. 

**Transformation**
Les codes/opérations Spark se composent essentiellement de transformations et d'actions. 

Transformation est une opération Spark qui lit un DataFrame (je prends df comme exemple), manipule des colonnes et éventuellement renvoie un autre DataFrame. Les exemples de transformation comprennent les opérations courantes : filtre, select, mean, count, tri, groupby, map, join, union, sample [écrire au tableau]…Vous avez bien compris, ce que vous avez l’habitude de faire durant la manipulation des données sont tous des transformations.

In [ ]:
myRange.where("id % 2 = 0")# filter sur les nombres paires
new_df = myRange.where("id % 2 = 0") 
new_df

# why there is no result => because this is a transformation (transformation is abstract), non execution sur data

new_df.show()
new_df.count()

In [ ]:
myRange.rdd.getNumPartitions()

new_df.rdd.getNumPartitions()

Comme ici, on est dans une autre dimension : on n’est plus dans une seule machine mais plusieurs, il faut distinguer deux types de transformations : 
Narrow transformations : chaque partition d'entrée ne contribue qu'à une seule partition de sortie. Chaque executor travaille individuellement, sans devoir communiquer avec les autres pour effectuer son opération : typiquement filtre ;

Wide transformation : Les partitions doivent être mélangées pour obtenir un résultat.
ie afin d’effectuer un calcul : 
Soit vous devez mélanger les partitions, une partition d'entrée contribue à plusieurs partitions de sortie (par forcément toutes les partitions. Cela dépend de comment vous avez partitionné vos df) : join
Soit vous devez rassembler les résultats à la fin : count / sum => ici c’est transformation mais pas action = > ne pas confondre filtre avec sum(), dans filtre, on n’a pas besoin de rassembler les partitions du DF filtré. Elles peuvent rester dans les différentes machines comme le DF initial. Mais sum() on doit agréger pour obtenir le résultat. 

Shuffle Opérations
Dans wide transformation il y a forcément Shuffle. Une Shuffle Operation est déclenchée lorsque des partitions des données doivent être mélangées donc concrètement être déplacées entre les exécutors (ie Spark va échanger des partitions à travers le cluster). Il s'agit d'un élément essentiel de nombreuses transformations, telles que .join().
tenance. 




In [ ]:
**Lazy Evaluation**

Maintenant, on va voir pourquoi la transformation ne donne pas de sortie  => ne déclenche pas d'exécution.

Les transformations sont évaluées en mode Lazy Evaluation. Cela signifie qu’il n’y a pas d’exécution. Autrement dit qu'aucun job Spark n'est déclenché par les transformations, quel que soit le nombre de transformations programmées. Pas de job=> pas d’exécution.

Lazy Evaluation est un modèle courant dans les langages de Big Data, par exemple PySpark, Scala, Java Stream API, python Dask etc. 

Si pas d’exécution, que fait-elle une transformation ? Elle construit un plan de logique d’exécution, si vous enchaînez les transformations, le plan logique va s’enchaîner. Si nous construisons un gros travail de data manipulation via Spark (map, union, sample…) mais que nous spécifions un filtre à la fin qui ne nécessite que l'extraction de quelques lignes uniquement, la manière la plus efficace d'exécuter ce travail est évidemment de faire le filter au début [schema au tableau]. Avec Spark, comme rien a été vraiment déclenché. Quand spark lit le clause “filtre”, il va se dire : OK, dans mon plan, je vais exécuter le filtre avant tout le reste. Et lors d’exécution, le filtre sera exécuté avant selon le plan. En R ou python pandas, on aurait déjà tout exécuté, c’est trop tard d’optimiser. En mode Lazy Evaluation, Spark optimisera cela pour nous dans son plan logique, en poussant le filtre au début (vers le bas sur le plan) automatiquement.

Au lieu de modifier les données immédiatement, Spark attendra le dernier moment pour optimiser et exécuter son plan logique. C’est quand le dernier moment ? Il va attendre une action qui déclenche l’exécution. Les avantages sont immenses, car Spark peut optimiser l'ensemble du flux de processing.

D’ailleurs, par exemple, vous avez des combinaisons de filtres complexes à mettre en place : les utilisateurs peuvent coder des petites transformations pour faciliter la lecture et la gestion. Mais Spark regroupe en interne ces transformations, en réduisant le nombre de passages sur les données. Spark cherche à tout prix de regrouper plusieurs transformations en une seule, afin de lire les données une seule fois pour appliquer les transformations en même temps, au lieu de les lire deux fois (exemple : enchaîner plusieurs filtres). En faisant cela vous gagnez aussi la visibilité et la facilité de gestion et de maintenance. 


**Action**

Les transformations permettent d'élaborer un plan logique (plan de modification de DataFrame). Pas d’exécution réelle. 

Pour déclencher le calcul des transformations, nous appelons une action : une opération Spark qui renvoie effectivement un résultat dans la console ou écrit sur le disque. 

Il existe peu d’action (qu’on peut compter avec 2 mains) : 
To console : 
.show() : afficher les données dans la console
.count()
.head(), .first(), .last() => console

To driver process : 
.take() : renvoie le nombre spécifié d'enregistrements au driver process
.collect() : collecte tous les résultats de tous les nœuds de travail et les renvoie au driver process. N'utilisez cette méthode que pour renvoyer de petites données agrégées.
.toPandas() : collecte tous les enregistrements de tous les travailleurs, les renvoie au driver process, puis convertit les résultats en un DataFrame pandas. N'utilisez cette méthode que pour renvoyer de petites données agrégées.



In [ ]:
myRange.where("id % 2 = 0").selectExpr('sum(id)').show() # 2 transformations 

In [ ]:
dataFrame1 = spark.range(2, 1000000, 2)
dataFrame2 = spark.range(2, 1000000, 4)

In [ ]:
df1 = dataFrame1.repartition(5) # stage 1, 5 partitions
df2 = dataFrame2.repartition(6) # stage 2, 6 partitions # nouveau stage car une autre branche du plan logique
df3 = df2.selectExpr("id * 5 as id") # stage 2, 6 partitions
df4 = df3.join(df1, ['id']) # stage 3, ?? partitions

In [ ]:
print(df1.rdd.getNumPartitions())
print(df2.rdd.getNumPartitions())
print(df3.rdd.getNumPartitions())
print(df4.rdd.getNumPartitions())

In [ ]:
spark.stop()